# 05 — Evaluation and Analysis

**RLHF Pipeline for Compact Open-Source LLMs**

This notebook provides the evaluation pipeline for comparing base, SFT, and PPO-aligned models. It supports the analysis required for all three research questions.

## Evaluation Dimensions

1. **Qualitative comparison** — side-by-side outputs from each model
2. **Automatic metrics** — response length, reward scores, win rates
3. **Verbosity bias analysis** (RQ3) — length trends before/after PPO
4. **Reward hacking indicators** (RQ3) — reward vs quality divergence
5. **Optional reference-based metrics** — BLEU/ROUGE if references available

## 5.1 Environment and Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")

from src.data_utils import set_seed, load_dataset_from_disk
from src.evaluation import (
    compute_response_lengths,
    compute_win_rate,
    evaluate_model_outputs,
    analyze_verbosity_bias,
    analyze_reward_hacking,
    save_evaluation_summary,
)
from src.metrics import (
    compute_diversity_metrics,
    length_reward_correlation,
)
from src.inference import generate_comparison_table, save_samples, load_samples
from src.reward_train import compute_reward_scores
from src.plotting import (
    plot_model_comparison_bar,
    plot_verbosity_comparison,
)

set_seed(42)
print("Imports OK")

In [ ]:
# Paths
SFT_MODEL_DIR    = PROJECT_ROOT / "outputs" / "models" / "sft_qwen"
REWARD_MODEL_DIR = PROJECT_ROOT / "outputs" / "models" / "reward_model_hh"
PPO_MODEL_DIR    = PROJECT_ROOT / "outputs" / "models" / "ppo_qwen"
LOG_DIR          = PROJECT_ROOT / "outputs" / "logs"
FIGURES_DIR      = PROJECT_ROOT / "outputs" / "figures"
TABLES_DIR       = PROJECT_ROOT / "outputs" / "tables"
SAMPLES_DIR      = PROJECT_ROOT / "outputs" / "samples"

for d in [FIGURES_DIR, TABLES_DIR, SAMPLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 5.2 Load Evaluation Prompts

In [ ]:
# Load held-out test prompts
test_data = load_dataset_from_disk(PROJECT_ROOT / "data" / "processed" / "hh_rlhf" / "test")
eval_prompts = test_data["prompt"][:50]  # Use 50 prompts for evaluation
print(f"Evaluation prompts: {len(eval_prompts)}")

## 5.3 Generate Responses from All Models

**TO BE FILLED AFTER ALL MODELS ARE TRAINED**

This cell loads each model, generates responses for the same prompts, and creates a comparison table.

In [ ]:
# This cell requires all models to be trained.
# Uncomment and run after completing notebooks 02, 03, 04.

# from src.model_utils import load_causal_lm, load_tokenizer, load_peft_model
# from src.inference import generate_responses
#
# MODEL_NAME = "Qwen/Qwen2.5-0.5B"
# tokenizer = load_tokenizer(MODEL_NAME)
# prompt_template = "### Human:\n{prompt}\n\n### Assistant:\n"
#
# # --- Base model ---
# base_model = load_causal_lm(MODEL_NAME, {"training": {"fp16": False, "bf16": False}})
# base_responses = generate_responses(base_model, tokenizer, eval_prompts,
#                                     max_new_tokens=128, prompt_template=prompt_template)
# del base_model
#
# # --- SFT model ---
# sft_model = load_peft_model(MODEL_NAME, SFT_MODEL_DIR,
#                              {"training": {"fp16": False, "bf16": False}})
# sft_responses = generate_responses(sft_model, tokenizer, eval_prompts,
#                                    max_new_tokens=128, prompt_template=prompt_template)
# del sft_model
#
# # --- PPO model ---
# # Load PPO model similarly (may need AutoModelForCausalLMWithValueHead)
# # ppo_responses = ...
#
# # Build comparison DataFrame
# records = []
# for i, prompt in enumerate(eval_prompts):
#     records.append({"prompt": prompt, "model_name": "base", "response": base_responses[i]})
#     records.append({"prompt": prompt, "model_name": "sft",  "response": sft_responses[i]})
#     # records.append({"prompt": prompt, "model_name": "ppo",  "response": ppo_responses[i]})
# comparison_df = pd.DataFrame(records)
# save_samples(comparison_df, SAMPLES_DIR / "model_comparison.csv")

print("TO BE FILLED AFTER ALL MODELS ARE TRAINED")

## 5.4 Load Existing Comparison Data (if available)

In [ ]:
comparison_path = SAMPLES_DIR / "model_comparison.csv"

if comparison_path.exists():
    comparison_df = load_samples(comparison_path)
    print(f"Loaded {len(comparison_df)} comparison rows")
    print(f"Models: {comparison_df['model_name'].unique()}")
else:
    print("No comparison data found. Run section 5.3 first.")
    comparison_df = None

## 5.5 Automatic Evaluation Metrics

In [ ]:
if comparison_df is not None:
    summary_df = evaluate_model_outputs(comparison_df)
    display(summary_df)
    summary_df.to_csv(TABLES_DIR / "evaluation_summary.csv", index=False)
    
    # Bar chart comparison
    for metric in ["length_mean_words", "length_mean_chars"]:
        if metric in summary_df.columns:
            plot_model_comparison_bar(
                summary_df, metric,
                title=f"Model Comparison: {metric}",
                output_path=FIGURES_DIR / f"comparison_{metric}.png",
            )
else:
    print("TO BE FILLED AFTER GENERATING MODEL COMPARISON DATA")

## 5.6 Reward-Score Based Win Rate

Compare SFT vs PPO using the reward model as a proxy judge.

In [ ]:
# Compute reward scores for each model's outputs and derive win rates.
# Requires: reward model loaded, comparison_df with SFT and PPO responses.

if comparison_df is not None and REWARD_MODEL_DIR.exists():
    # Load reward model for scoring
    # rm = load_peft_model(MODEL_NAME, REWARD_MODEL_DIR, rm_cfg, is_reward_model=True)
    # rm.eval()
    # rm_tok = load_tokenizer(MODEL_NAME)
    #
    # sft_rows = comparison_df[comparison_df["model_name"] == "sft"]
    # ppo_rows = comparison_df[comparison_df["model_name"] == "ppo"]
    # sft_texts = [f"{p}\n\n{r}" for p, r in zip(sft_rows["prompt"], sft_rows["response"])]
    # ppo_texts = [f"{p}\n\n{r}" for p, r in zip(ppo_rows["prompt"], ppo_rows["response"])]
    # sft_scores = compute_reward_scores(rm, rm_tok, sft_texts, 512)
    # ppo_scores = compute_reward_scores(rm, rm_tok, ppo_texts, 512)
    # win_rate = compute_win_rate(ppo_scores, sft_scores, "PPO", "SFT")
    # print(win_rate)
    print("TO BE FILLED AFTER FULL EXPERIMENTS")
else:
    print("TO BE FILLED AFTER FULL EXPERIMENTS")

## 5.7 Verbosity Bias Analysis (RQ3)

Does the PPO-aligned model produce systematically longer responses?

In [ ]:
if comparison_df is not None:
    sft_rows = comparison_df[comparison_df["model_name"] == "sft"]
    ppo_rows = comparison_df[comparison_df["model_name"] == "ppo"] if "ppo" in comparison_df["model_name"].values else None
    
    if ppo_rows is not None and len(ppo_rows) > 0:
        verbosity = analyze_verbosity_bias(
            sft_rows["response"].tolist(),
            ppo_rows["response"].tolist(),
        )
        for k, v in verbosity.items():
            print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")
        
        # Plot
        sft_lens = [len(r.split()) for r in sft_rows["response"]]
        ppo_lens = [len(r.split()) for r in ppo_rows["response"]]
        plot_verbosity_comparison(
            sft_lens, ppo_lens,
            output_path=FIGURES_DIR / "verbosity_comparison.png",
        )
    else:
        print("PPO responses not yet available")
else:
    print("TO BE FILLED AFTER GENERATING MODEL COMPARISON DATA")

## 5.8 Reward Hacking Indicators (RQ3)

Analyse PPO training logs for signs of reward hacking.

In [ ]:
ppo_log_path = LOG_DIR / "ppo_training_log.csv"

if ppo_log_path.exists():
    ppo_df = pd.read_csv(ppo_log_path)
    indicators = analyze_reward_hacking(ppo_df)
    
    print("Reward hacking indicators:")
    for k, v in indicators.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    
    pd.DataFrame([indicators]).to_csv(TABLES_DIR / "reward_hacking_indicators.csv", index=False)
else:
    print("TO BE FILLED AFTER RUNNING PPO TRAINING")

## 5.9 Response Diversity Analysis

In [ ]:
if comparison_df is not None:
    for model_name in comparison_df["model_name"].unique():
        responses = comparison_df[comparison_df["model_name"] == model_name]["response"].tolist()
        diversity = compute_diversity_metrics(responses)
        print(f"\n{model_name}:")
        for k, v in diversity.items():
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
else:
    print("TO BE FILLED AFTER GENERATING MODEL COMPARISON DATA")

## 5.10 Qualitative Comparison Table

**TO BE FILLED AFTER RUNNING FULL EXPERIMENTS**

Select 5-10 interesting prompts and present model outputs side by side.

In [ ]:
if comparison_df is not None:
    # Pivot to wide format for side-by-side comparison
    pivot = comparison_df.pivot_table(
        index="prompt", columns="model_name", values="response", aggfunc="first"
    ).reset_index()
    
    # Show first 5
    for idx, row in pivot.head(5).iterrows():
        print(f"\nPROMPT: {row['prompt'][:150]}")
        for col in pivot.columns:
            if col != "prompt":
                val = row[col] if pd.notna(row[col]) else "N/A"
                print(f"  [{col}]: {str(val)[:200]}")
        print("-" * 80)
else:
    print("TO BE FILLED AFTER GENERATING MODEL COMPARISON DATA")

## Summary

| Output | Location |
|---|---|
| Model comparison CSV | `outputs/samples/model_comparison.csv` |
| Evaluation summary | `outputs/tables/evaluation_summary.csv` |
| Reward hacking indicators | `outputs/tables/reward_hacking_indicators.csv` |
| Verbosity comparison plot | `outputs/figures/verbosity_comparison.png` |
| Metric comparison plots | `outputs/figures/comparison_*.png` |

**Next:** Proceed to `06_human_eval_template.ipynb` for human evaluation (optional).